In [3]:
# MODELING 00 — ENVIRONMENT DIAGNOSTIC (FIXED)
# ============================================================

import sys
import pandas as pd
import numpy as np
import sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 60)
print("MODELING 00 — ENVIRONMENT DIAGNOSTIC")
print("=" * 60)

print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Pandas version    :", pd.__version__)
print("NumPy version     :", np.__version__)
print("Scikit-learn      :", sklearn.__version__)

print("\nLibrary import validation:")
print("Pandas            : PASS")
print("NumPy             : PASS")
print("Scikit-learn      : PASS")
print("TF-IDF import     : PASS")
print("Cosine similarity : PASS")

print("\nMODELING ENVIRONMENT: PASS")


MODELING 00 — ENVIRONMENT DIAGNOSTIC
Python executable : c:\Penelitian Sentimen Bencana Nasional\.venv\Scripts\python.exe
Python version    : 3.14.8
Pandas version    : 3.0.6
NumPy version     : 2.5.3
Scikit-learn      : 1.9.1

Library import validation:
Pandas            : PASS
NumPy             : PASS
Scikit-learn      : PASS
TF-IDF import     : PASS
Cosine similarity : PASS

MODELING ENVIRONMENT: PASS


In [4]:
# MODELING 01 — MODELING PREFLIGHT & DATASET INTEGRITY
# 348 LABELED TWEETS | READ-ONLY SOURCE
#
# Checks:
# 01. Dataset provenance / SHA-256
# 02. Schema, IDs, labels, metadata
# 03. Text length / missing data
# 04. Class and disaster-case distribution
# 05. Exact duplicate checks
# 06. Near-duplicate similarity audit
# 07. Stratified split feasibility
#
# No tokenization, no splitting, no training.
# ============================================================

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# A. PROJECT CONFIGURATION
# ============================================================

ROOT = Path(
    r"C:\Penelitian Sentimen Bencana Nasional"
)

DATA_PATH = (
    ROOT / "03_Labeling"
    / "outputs" / "final_labeling"
    / "modeling_labeled_348.csv"
)

AUDIT_SOURCE_PATH = (
    ROOT / "03_Labeling"
    / "outputs" / "final_labeling"
    / "final_labeling_audit.json"
)

MASTER_FINAL_PATH = (
    ROOT / "03_Labeling"
    / "outputs" / "final_labeling"
    / "labeled_final_381.csv"
)

OUTPUT_DIR = (
    ROOT / "04_Modeling"
    / "outputs" / "preflight"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_PATH = (
    OUTPUT_DIR / "modeling_preflight_report.json"
)

NEAR_DUP_PATH = (
    OUTPUT_DIR / "near_duplicate_candidates.csv"
)

TEXT_STATS_PATH = (
    OUTPUT_DIR / "modeling_text_statistics.csv"
)

# Immutable source version from Part17 GitHub commit.
EXPECTED_SHA256 = (
    "f3ba0bbffe98140e38270b666e2d4fbe21342f57d40c0b2d4ad70335caf88bfd"
)

EXPECTED_ROWS = 348
EXPECTED_FINAL_ROWS = 381

VALID_LABELS = {
    "POSITIF",
    "NEGATIF",
    "NETRAL"
}

VALID_CASES = {
    "lombok_2018",
    "palu_2018",
    "sumatra_2025"
}

# This threshold generates REVIEW CANDIDATES.
# It does not define confirmed duplicate tweets.
NEAR_DUP_THRESHOLD = 0.80

# Modeling is not yet allowed in this cell.
TRAINING_ALLOWED = False

# ============================================================
# B. HELPER FUNCTIONS
# ============================================================

def sha256_file(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def clean_string(series):
    return (
        series
        .astype("string")
        .fillna("")
        .str.strip()
    )


def distribution(series):
    return {
        str(k): int(v)
        for k, v in series.value_counts().items()
    }


def safe_ratio(count, total):
    return round(
        (count / total * 100) if total else 0,
        4
    )


def normalize_similarity_text(value):
    value = str(value).lower()
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def ensure_new_outputs():
    outputs = [
        REPORT_PATH,
        NEAR_DUP_PATH,
        TEXT_STATS_PATH
    ]

    existing = [
        str(p) for p in outputs if p.exists()
    ]

    if existing:
        raise FileExistsError(
            "Preflight output already exists. "
            "Do not overwrite without review:\n"
            + "\n".join(existing)
        )


# ============================================================
# C. SOURCE VALIDATION
# ============================================================

print("=" * 70)
print("MODELING 01 — PREFLIGHT & DATASET INTEGRITY")
print("=" * 70)

assert DATA_PATH.is_file(), (
    f"Missing dataset: {DATA_PATH}"
)

assert AUDIT_SOURCE_PATH.is_file(), (
    f"Missing audit file: {AUDIT_SOURCE_PATH}"
)

assert MASTER_FINAL_PATH.is_file(), (
    f"Missing final dataset: {MASTER_FINAL_PATH}"
)

ensure_new_outputs()

source_hashes_before = {
    "modeling_348": sha256_file(DATA_PATH),
    "final_381": sha256_file(MASTER_FINAL_PATH),
    "labeling_audit": sha256_file(AUDIT_SOURCE_PATH)
}

audit_source = json.loads(
    AUDIT_SOURCE_PATH.read_text(
        encoding="utf-8"
    )
)

assert source_hashes_before[
    "modeling_348"
] == EXPECTED_SHA256, (
    "Dataset hash differs from Part17. STOP."
)

assert audit_source[
    "modeling_file_sha256"
] == EXPECTED_SHA256

assert audit_source["model_candidates"] == 348
assert audit_source["final_labeled_rows"] == 381

assert audit_source[
    "duplicate_conflicts_remaining"
] == 0

assert audit_source["status"] == (
    "READY_FOR_MODELING_PREPARATION"
)

df = pd.read_csv(
    DATA_PATH,
    dtype={"id": "string"},
    encoding="utf-8-sig",
    keep_default_na=False
)

df_final = pd.read_csv(
    MASTER_FINAL_PATH,
    dtype={"id": "string"},
    encoding="utf-8-sig",
    keep_default_na=False
)

print("SOURCE HASH: PASS")
print("PART17 AUDIT STATUS: PASS")

# ============================================================
# D. DATASET SCHEMA & INTEGRITY
# ============================================================

required_columns = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "duplicate_group_id",
    "final_text_hash",
    "sentiment_label",
    "annotation_status",
    "label_notes",
    "annotator",
    "labeled_at"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

assert len(df) == EXPECTED_ROWS
assert len(df_final) == EXPECTED_FINAL_ROWS

df["id"] = clean_string(df["id"])

assert df["id"].ne("").all()
assert df["id"].is_unique

assert df_final["id"].astype(
    "string"
).str.strip().is_unique

assert set(df["id"]).issubset(
    set(df_final["id"].astype("string").str.strip())
)

for col in [
    "case",
    "sentiment_label",
    "annotation_status"
]:
    df[col] = clean_string(df[col])

df["sentiment_label"] = (
    df["sentiment_label"].str.upper()
)

df["annotation_status"] = (
    df["annotation_status"].str.upper()
)

assert df["case"].isin(VALID_CASES).all()
assert df["sentiment_label"].isin(
    VALID_LABELS
).all()

assert df["annotation_status"].eq(
    "LABELED"
).all()

for col in [
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "duplicate_group_id",
    "final_text_hash",
    "label_notes",
    "annotator",
    "labeled_at"
]:
    df[col] = clean_string(df[col])

    assert df[col].ne("").all(), (
        f"Missing/empty values: {col}"
    )

assert df["final_text_hash"].is_unique

assert df["duplicate_group_id"].is_unique

assert df["case"].nunique() == 3
assert df["sentiment_label"].nunique() == 3

# Validate final labels using ID joins.
final_labels = df_final.set_index("id")[
    "sentiment_label"
].astype("string").str.strip().str.upper()

model_labels = (
    df.set_index("id")["sentiment_label"]
    .sort_index()
)

assert model_labels.equals(
    final_labels.loc[
        model_labels.index
    ].sort_index()
), "348 labels differ from final adjudicated labels."

print("ROW COUNT: 348 / 348")
print("UNIQUE TWEET IDs: PASS")
print("UNIQUE TEXT HASHES: PASS")
print("UNIQUE DUPLICATE GROUPS: PASS")
print("ALL LABELS & METADATA: PASS")

# ============================================================
# E. TEXT QUALITY STATISTICS
# ============================================================

# No new cleaning transformation is applied.
# These statistics only inspect existing text.

text_fields = [
    "text_original",
    "text_semantic_verified",
    "text_linguistic"
]

for field in text_fields:
    df[f"{field}_chars"] = (
        df[field].str.len()
    )

    df[f"{field}_words"] = (
        df[field]
        .str.split()
        .str.len()
    )

df["model_text_chars"] = (
    df["text_linguistic_chars"]
)

df["model_text_words"] = (
    df["text_linguistic_words"]
)

text_summary = {}

for field in text_fields:
    lengths = df[f"{field}_words"]

    text_summary[field] = {
        "min_words": int(lengths.min()),
        "mean_words": round(
            float(lengths.mean()), 2
        ),
        "median_words": round(
            float(lengths.median()), 2
        ),
        "p90_words": round(
            float(lengths.quantile(0.90)), 2
        ),
        "p95_words": round(
            float(lengths.quantile(0.95)), 2
        ),
        "max_words": int(lengths.max())
    }

text_stats_columns = [
    "case",
    "id",
    "sentiment_label",
    "final_text_hash",
    "text_original_chars",
    "text_original_words",
    "text_semantic_verified_chars",
    "text_semantic_verified_words",
    "text_linguistic_chars",
    "text_linguistic_words"
]

text_stats = df[text_stats_columns].copy()

print("\nTEXT LENGTH SUMMARY")

display(
    pd.DataFrame(text_summary).T
)

print("\nSHORTEST 10 MODEL TEXTS")

display(
    df.nsmallest(
        10,
        "model_text_words"
    )[
        [
            "case",
            "id",
            "sentiment_label",
            "model_text_words",
            "text_linguistic"
        ]
    ]
)

# ============================================================
# F. LABEL DISTRIBUTION
# ============================================================

label_counts = distribution(
    df["sentiment_label"]
)

case_counts = distribution(
    df["case"]
)

class_table = pd.DataFrame([
    {
        "sentiment": label,
        "count": label_counts.get(label, 0),
        "percentage": safe_ratio(
            label_counts.get(label, 0),
            len(df)
        )
    }
    for label in ["POSITIF", "NEGATIF", "NETRAL"]
])

print("\nCLASS DISTRIBUTION")
display(class_table)

case_label_table = pd.crosstab(
    df["case"],
    df["sentiment_label"]
)

print("\nCLASS DISTRIBUTION BY CASE")
display(case_label_table)

assert label_counts == {
    "NETRAL": 223,
    "POSITIF": 110,
    "NEGATIF": 15
}, "Class distribution differs from Part17."

assert case_counts == {
    "sumatra_2025": 142,
    "palu_2018": 105,
    "lombok_2018": 101
}, "Case distribution differs from Part17."

majority_baseline_accuracy = (
    max(label_counts.values()) / len(df)
)

print(
    "\nMajority-class baseline accuracy:",
    f"{majority_baseline_accuracy:.2%}"
)

# ============================================================
# G. EXACT DUPLICATE & MODELING INPUT AUDIT
# ============================================================

exact_hash_duplicates = int(
    df["final_text_hash"].duplicated().sum()
)

exact_group_duplicates = int(
    df["duplicate_group_id"].duplicated().sum()
)

normalized_model_text = (
    df["text_linguistic"]
    .map(normalize_similarity_text)
)

normalized_duplicate_rows = int(
    normalized_model_text.duplicated().sum()
)

assert exact_hash_duplicates == 0
assert exact_group_duplicates == 0

print("\nEXACT DUPLICATE AUDIT")
print("Duplicate hashes      :", exact_hash_duplicates)
print("Duplicate groups      :", exact_group_duplicates)
print(
    "Duplicate model texts :",
    normalized_duplicate_rows
)

# Even if model text duplicates exist, do not silently
# remove any row. Report them for investigation.

# ============================================================
# H. NEAR-DUPLICATE SIMILARITY AUDIT
# ============================================================

# Character n-grams are suitable for detecting overlap
# caused by minor typos, spacing and punctuation.
# A similarity score is not semantic equivalence.

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    lowercase=True,
    min_df=1,
    norm="l2"
)

X_similarity = vectorizer.fit_transform(
    normalized_model_text.tolist()
)

similarity_matrix = cosine_similarity(
    X_similarity
)

np.fill_diagonal(
    similarity_matrix,
    0.0
)

near_pairs = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        score = float(
            similarity_matrix[i, j]
        )

        if score < NEAR_DUP_THRESHOLD:
            continue

        a = df.iloc[i]
        b = df.iloc[j]

        near_pairs.append({
            "id_a": str(a["id"]),
            "id_b": str(b["id"]),
            "case_a": str(a["case"]),
            "case_b": str(b["case"]),
            "label_a": str(a["sentiment_label"]),
            "label_b": str(b["sentiment_label"]),
            "same_label": bool(
                a["sentiment_label"]
                == b["sentiment_label"]
            ),
            "similarity": round(score, 6),
            "text_a": str(a["text_linguistic"]),
            "text_b": str(b["text_linguistic"]),
            "semantic_text_a": str(
                a["text_semantic_verified"]
            ),
            "semantic_text_b": str(
                b["text_semantic_verified"]
            ),
            "review_decision": "",
            "review_notes": ""
        })

near_columns = [
    "id_a",
    "id_b",
    "case_a",
    "case_b",
    "label_a",
    "label_b",
    "same_label",
    "similarity",
    "text_a",
    "text_b",
    "semantic_text_a",
    "semantic_text_b",
    "review_decision",
    "review_notes"
]

near_df = pd.DataFrame(
    near_pairs,
    columns=near_columns
)

if not near_df.empty:
    near_df = near_df.sort_values(
        "similarity",
        ascending=False
    ).reset_index(drop=True)

near_different_labels = int(
    (~near_df["same_label"]).sum()
) if len(near_df) else 0

print("\nNEAR-DUPLICATE AUDIT")
print("Threshold              :", NEAR_DUP_THRESHOLD)
print("Candidate pairs        :", len(near_df))
print(
    "Pairs with label differences:",
    near_different_labels
)

if len(near_df):
    display(
        near_df[
            [
                "id_a",
                "id_b",
                "case_a",
                "case_b",
                "label_a",
                "label_b",
                "same_label",
                "similarity"
            ]
        ].head(30)
    )
else:
    print(
        "No near-duplicate candidates "
        "at the current threshold."
    )

# ============================================================
# I. STRATIFIED SPLIT FEASIBILITY
# ============================================================

# Feasibility only: no actual data splitting.

minority_count = min(
    label_counts.values()
)

requested_folds = 5

stratified_5fold_possible = (
    minority_count >= requested_folds
)

estimated_minority_per_fold = (
    minority_count / requested_folds
)

split_feasibility = {
    "total_rows": 348,
    "minimum_class_count": int(minority_count),
    "requested_folds": 5,
    "stratified_5fold_possible": bool(
        stratified_5fold_possible
    ),
    "estimated_negative_test_per_fold": (
        estimated_minority_per_fold
    ),
    "holdout_70_15_15": {
        "train_approx": 244,
        "validation_approx": 52,
        "test_approx": 52,
        "warning": (
            "NEGATIF approximately 2-3 samples "
            "per small holdout partition."
        )
    },
    "leakage_groups_reviewed": False,
    "split_created": False,
    "training_allowed": False
}

print("\nSPLIT FEASIBILITY")
print(
    "Stratified 5-Fold technically possible:",
    stratified_5fold_possible
)
print(
    "Smallest class:",
    minority_count
)
print(
    "Approx minority samples per test fold:",
    estimated_minority_per_fold
)

print(
    "Note: Fold feasibility does not guarantee "
    "statistically reliable model estimates."
)

# ============================================================
# J. DATASET DECISION GATES
# ============================================================

warnings = []

if normalized_duplicate_rows:
    warnings.append(
        "Duplicate normalized text_linguistic "
        "values need investigation."
    )

if len(near_df):
    warnings.append(
        f"{len(near_df)} near-duplicate candidate "
        "pairs require leakage review."
    )

if near_different_labels:
    warnings.append(
        f"{near_different_labels} similar-text pairs "
        "have different sentiment labels."
    )

if label_counts["NEGATIF"] < 20:
    warnings.append(
        "NEGATIF has fewer than 20 examples. "
        "Per-class evaluation will be unstable."
    )

# The handoff is valid but split strategy
# must still be explicitly defined.
DATASET_INTEGRITY_PASS = True
TRAINING_ALLOWED = False

status = "READY_FOR_SPLIT_DESIGN"

# ============================================================
# K. PREPARE PREFLIGHT REPORT
# ============================================================

report = {
    "stage": "04_Modeling",
    "part": "01 Modeling Preflight",
    "generated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "source_commit": (
        "905d35995be984a75dc38ec4687db61fe37c125f"
    ),
    "source_hashes": source_hashes_before,
    "total_rows": int(len(df)),
    "unique_ids": int(df["id"].nunique()),
    "label_distribution": label_counts,
    "case_distribution": case_counts,
    "text_summary": text_summary,
    "exact_hash_duplicate_rows": (
        exact_hash_duplicates
    ),
    "exact_group_duplicate_rows": (
        exact_group_duplicates
    ),
    "normalized_model_text_duplicate_rows": (
        normalized_duplicate_rows
    ),
    "near_duplicate_threshold": (
        NEAR_DUP_THRESHOLD
    ),
    "near_duplicate_candidate_pairs": (
        len(near_df)
    ),
    "near_duplicate_different_label_pairs": (
        near_different_labels
    ),
    "majority_baseline_accuracy": round(
        majority_baseline_accuracy,
        6
    ),
    "split_feasibility": split_feasibility,
    "warnings": warnings,
    "dataset_integrity_pass": (
        DATASET_INTEGRITY_PASS
    ),
    "split_created": False,
    "training_started": False,
    "training_allowed": False,
    "status": status
}

# ============================================================
# L. VERIFY INPUTS STILL UNCHANGED
# ============================================================

source_hashes_after = {
    "modeling_348": sha256_file(DATA_PATH),
    "final_381": sha256_file(MASTER_FINAL_PATH),
    "labeling_audit": sha256_file(AUDIT_SOURCE_PATH)
}

assert source_hashes_before == source_hashes_after, (
    "Input files changed during preflight."
)

# ============================================================
# M. EXPORT REPORTS
# ============================================================

# No source CSV or final annotation is overwritten.

text_stats.to_csv(
    TEXT_STATS_PATH,
    index=False,
    encoding="utf-8-sig"
)

near_df.to_csv(
    NEAR_DUP_PATH,
    index=False,
    encoding="utf-8-sig"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

assert source_hashes_before == {
    "modeling_348": sha256_file(DATA_PATH),
    "final_381": sha256_file(MASTER_FINAL_PATH),
    "labeling_audit": sha256_file(AUDIT_SOURCE_PATH)
}

# ============================================================
# N. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("MODELING 01 — PREFLIGHT SUMMARY")
print("=" * 70)

print("Total model records :", len(df))
print("Unique IDs          :", df["id"].nunique())
print("Class distribution  :", label_counts)
print("Case distribution   :", case_counts)
print("Exact hash duplicate:", exact_hash_duplicates)
print("Near-duplicate pairs:", len(near_df))
print("Label disagreements :", near_different_labels)
print("Warnings            :", len(warnings))

print("\nOutput files:")
print("Preflight report:", REPORT_PATH)
print("Text statistics :", TEXT_STATS_PATH)
print("Near duplicates :", NEAR_DUP_PATH)

print("\nDATASET INTEGRITY: PASS")
print("SPLIT DESIGN: PENDING")
print("TRAINING: HOLD")
print("MODELING 01 PREFLIGHT: PASS")
print("=" * 70)


MODELING 01 — PREFLIGHT & DATASET INTEGRITY
SOURCE HASH: PASS
PART17 AUDIT STATUS: PASS
ROW COUNT: 348 / 348
UNIQUE TWEET IDs: PASS
UNIQUE TEXT HASHES: PASS
UNIQUE DUPLICATE GROUPS: PASS
ALL LABELS & METADATA: PASS

TEXT LENGTH SUMMARY


,min_words,mean_words,median_words,p90_words,p95_words,max_words
text_original,6.0,39.10,27.0,49.3,107.65,804.0
text_semantic_verified,5.0,37.72,26.0,46.3,107.00,804.0
text_linguistic,5.0,37.72,26.0,46.3,107.00,804.0



SHORTEST 10 MODEL TEXTS


,case,id,sentiment_label,model_text_words,text_linguistic
283,lombok_2018,1027806753394708482,POSITIF,5,jadikan gempa lombok bencana nasional
9,lombok_2018,1026390946693271553,POSITIF,6,gempa ntb diminta jadi bencana nasional
209,lombok_2018,1028681595203379201,NETRAL,6,pemerintah: gempa lombok bukan bencana nasional
34,lombok_2018,1028974526841815041,NETRAL,7,live! gempa lombok bencana nasional - bersama
75,lombok_2018,1028857865086414848,NETRAL,7,pemerintah putuskan gempa lombok bukan bencana...
172,palu_2018,1046625007495917568,NETRAL,7,pemerintah tidak tetapkan gempa-tsunami palu b...
246,lombok_2018,1028311588921520134,NETRAL,7,dpr minta gempa lombok jadi bencana nasional
272,lombok_2018,1027601912437194753,NETRAL,7,legislator minta gempa lombok jadi bencana nas...
290,lombok_2018,1027894842997653505,NETRAL,7,dprd desak status gempa lombok bencana nasional
4,lombok_2018,1026421529775493121,POSITIF,8,pemerintah diminta naikan status lombok jadi b...



CLASS DISTRIBUTION


...


CLASS DISTRIBUTION BY CASE


sentiment_label,NEGATIF,NETRAL,POSITIF
case,,,
lombok_2018,3,64,34
palu_2018,5,70,30
sumatra_2025,7,89,46



Majority-class baseline accuracy: 64.08%

EXACT DUPLICATE AUDIT
Duplicate hashes      : 0
Duplicate groups      : 0
Duplicate model texts : 0

NEAR-DUPLICATE AUDIT
Threshold              : 0.8
Candidate pairs        : 12
Pairs with label differences: 3


,id_a,id_b,case_a,case_b,label_a,label_b,same_label,similarity
0,1996259280018948512,1996241105206325553,sumatra_2025,sumatra_2025,NETRAL,NETRAL,True,0.961614
1,1996210311628751167,1996149775524376777,sumatra_2025,sumatra_2025,NETRAL,NETRAL,True,0.947070
2,1047236117546774528,1046697758038446080,palu_2018,palu_2018,NETRAL,NETRAL,True,0.943492
3,1046688959743442944,1046688954014031872,palu_2018,palu_2018,NETRAL,NETRAL,True,0.943239
4,1027727426883805184,1027914286184550402,lombok_2018,lombok_2018,NETRAL,POSITIF,False,0.939963
5,1995776900560748859,1995870576632586438,sumatra_2025,sumatra_2025,NETRAL,NETRAL,True,0.918162
6,1028838553344663553,1028538923943776257,lombok_2018,lombok_2018,NETRAL,NETRAL,True,0.900152
7,1028857865086414848,1028560120047841280,lombok_2018,lombok_2018,NETRAL,NETRAL,True,0.880175
8,1046691986596495360,1046787140594618373,palu_2018,palu_2018,NETRAL,POSITIF,False,0.868881
9,1046064035505078274,1046217614148624387,palu_2018,palu_2018,POSITIF,NETRAL,False,0.868309



SPLIT FEASIBILITY
Stratified 5-Fold technically possible: True
Smallest class: 15
Approx minority samples per test fold: 3.0
Note: Fold feasibility does not guarantee statistically reliable model estimates.

MODELING 01 — PREFLIGHT SUMMARY
Total model records : 348
Unique IDs          : 348
Class distribution  : {'NETRAL': 223, 'POSITIF': 110, 'NEGATIF': 15}
Case distribution   : {'sumatra_2025': 142, 'palu_2018': 105, 'lombok_2018': 101}
Exact hash duplicate: 0
Near-duplicate pairs: 12
Label disagreements : 3
Warnings            : 3

Output files:
Preflight report: C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\preflight\modeling_preflight_report.json
Text statistics : C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\preflight\modeling_text_statistics.csv
Near duplicates : C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\preflight\near_duplicate_candidates.csv

DATASET INTEGRITY: PASS
SPLIT DESIGN: PENDING
TRAINING: HOLD
MODELING 01 PREFLIGHT: PASS


In [5]:
# MODELING 02 — NEAR-DUPLICATE REVIEW PREPARATION
# 348 TWEETS | 12 PAIRS | 3 LABEL DISAGREEMENTS
#
# READ ONLY:
# - modeling_labeled_348.csv
# - near_duplicate_candidates.csv
# - modeling_preflight_report.json
#
# NEW OUTPUT ONLY:
# - near_duplicate_review_12.xlsx
# - near_duplicate_snapshot.csv
# - near_duplicate_review_manifest.json
#
# NO RELABELING | NO DELETION | NO SPLITTING
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)
from openpyxl.worksheet.datavalidation import DataValidation

# ============================================================
# A. PATH CONFIGURATION
# ============================================================

ROOT = Path(
    r"C:\Penelitian Sentimen Bencana Nasional"
)

MODEL_DIR = ROOT / "04_Modeling"

SOURCE_DATA = (
    ROOT / "03_Labeling"
    / "outputs" / "final_labeling"
    / "modeling_labeled_348.csv"
)

PREFLIGHT_DIR = (
    MODEL_DIR / "outputs" / "preflight"
)

NEAR_SOURCE = (
    PREFLIGHT_DIR / "near_duplicate_candidates.csv"
)

PREFLIGHT_REPORT = (
    PREFLIGHT_DIR / "modeling_preflight_report.json"
)

OUTPUT_DIR = (
    MODEL_DIR / "outputs"
    / "near_duplicate_review"
)

REVIEW_PATH = (
    OUTPUT_DIR / "near_duplicate_review_12.xlsx"
)

SNAPSHOT_PATH = (
    OUTPUT_DIR / "near_duplicate_snapshot.csv"
)

MANIFEST_PATH = (
    OUTPUT_DIR / "near_duplicate_review_manifest.json"
)

EXPECTED_ROWS = 348
EXPECTED_PAIRS = 12
EXPECTED_DIFFERENT_LABEL_PAIRS = 3

EXPECTED_DATA_HASH = (
    "f3ba0bbffe98140e38270b666e2d4fbe21342f57d40c0b2d4ad70335caf88bfd"
)

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}

# ============================================================
# B. HELPER FUNCTIONS
# ============================================================

def sha256_file(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def normalize_id(series):
    result = series.astype("string").str.strip()

    assert result.notna().all()
    assert result.ne("").all()

    return result


def normalize_string(series):
    return (
        series.astype("string")
        .fillna("")
        .str.strip()
    )


def require_columns(frame, columns, name):
    missing = [
        col for col in columns
        if col not in frame.columns
    ]

    assert not missing, (
        f"{name} missing columns: {missing}"
    )


# ============================================================
# C. SAFETY CHECK
# ============================================================

print("=" * 70)
print("MODELING 02 — NEAR-DUPLICATE REVIEW PREPARATION")
print("=" * 70)

required_inputs = [
    SOURCE_DATA,
    NEAR_SOURCE,
    PREFLIGHT_REPORT
]

for path in required_inputs:
    assert path.is_file(), (
        f"Input file not found: {path}"
    )

# Check before creating output directory or files.
for path in [
    REVIEW_PATH,
    SNAPSHOT_PATH,
    MANIFEST_PATH
]:
    if path.exists():
        raise FileExistsError(
            "Output already exists. "
            "Do not overwrite manual review:\n"
            f"{path}"
        )

source_hashes_before = {
    "modeling_dataset": sha256_file(SOURCE_DATA),
    "near_duplicate_source": sha256_file(NEAR_SOURCE),
    "preflight_report": sha256_file(PREFLIGHT_REPORT)
}

assert (
    source_hashes_before["modeling_dataset"]
    == EXPECTED_DATA_HASH
), "The 348 dataset differs from Part17."

preflight = json.loads(
    PREFLIGHT_REPORT.read_text(encoding="utf-8")
)

assert preflight["dataset_integrity_pass"] is True
assert preflight["total_rows"] == 348
assert preflight["near_duplicate_candidate_pairs"] == 12
assert (
    preflight["near_duplicate_different_label_pairs"]
    == 3
)

assert preflight["training_allowed"] is False

assert preflight["source_hashes"]["modeling_348"] == (
    source_hashes_before["modeling_dataset"]
)

print("SOURCE INTEGRITY: PASS")
print("PREFLIGHT MANIFEST: PASS")
print("NO OUTPUT OVERWRITE: PASS")

# ============================================================
# D. LOAD DATA
# ============================================================

df = pd.read_csv(
    SOURCE_DATA,
    dtype={"id": "string"},
    encoding="utf-8-sig",
    keep_default_na=False
)

near = pd.read_csv(
    NEAR_SOURCE,
    dtype={
        "id_a": "string",
        "id_b": "string"
    },
    encoding="utf-8-sig",
    keep_default_na=False
)

required_model_cols = [
    "id",
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "sentiment_label",
    "label_notes",
    "final_text_hash",
    "duplicate_group_id"
]

required_near_cols = [
    "id_a",
    "id_b",
    "case_a",
    "case_b",
    "label_a",
    "label_b",
    "same_label",
    "similarity",
    "text_a",
    "text_b",
    "semantic_text_a",
    "semantic_text_b"
]

require_columns(df, required_model_cols, "model dataset")
require_columns(near, required_near_cols, "near candidates")

df["id"] = normalize_id(df["id"])
near["id_a"] = normalize_id(near["id_a"])
near["id_b"] = normalize_id(near["id_b"])

assert len(df) == EXPECTED_ROWS
assert df["id"].is_unique

assert len(near) == EXPECTED_PAIRS

assert df["sentiment_label"].isin(
    VALID_LABELS
).all()

assert df["final_text_hash"].is_unique
assert df["duplicate_group_id"].is_unique

# Detect duplicate pairs regardless of A/B order.
near["pair_key"] = near.apply(
    lambda row: "||".join(sorted([
        str(row["id_a"]),
        str(row["id_b"])
    ])),
    axis=1
)

assert near["pair_key"].is_unique
assert (near["id_a"] != near["id_b"]).all()

all_ids = set(df["id"])

assert set(near["id_a"]).issubset(all_ids)
assert set(near["id_b"]).issubset(all_ids)

near["similarity"] = pd.to_numeric(
    near["similarity"],
    errors="raise"
)

assert near["similarity"].between(
    0.80 - 1e-6, 1.0 + 1e-6
).all()

print("MODEL DATA: 348 ROWS — PASS")
print("NEAR-DUPLICATE PAIRS: 12 — PASS")

# ============================================================
# E. VERIFY PAIRS AGAINST FINAL DATA
# ============================================================

lookup = df.set_index("id")

# Confirm stored fields belong to the stated tweet IDs.
for side in ["a", "b"]:

    ids = near[f"id_{side}"]

    for near_col, source_col in [
        (f"case_{side}", "case"),
        (f"label_{side}", "sentiment_label"),
        (f"text_{side}", "text_linguistic"),
        (
            f"semantic_text_{side}",
            "text_semantic_verified"
        )
    ]:
        expected = ids.map(lookup[source_col])

        actual = near[near_col]

        assert normalize_string(actual).reset_index(
            drop=True
        ).equals(
            normalize_string(expected).reset_index(
                drop=True
            )
        ), (
            f"Source mismatch: {near_col}"
        )

label_disagreement = (
    near["label_a"] != near["label_b"]
)

assert int(label_disagreement.sum()) == (
    EXPECTED_DIFFERENT_LABEL_PAIRS
)

print("PAIR-TO-SOURCE VALIDATION: PASS")
print("DIFFERENT LABEL PAIRS: 3")

# ============================================================
# F. PREPARE SNAPSHOT AND REVIEW DATA
# ============================================================

near = near.sort_values(
    ["similarity", "pair_key"],
    ascending=[False, True]
).reset_index(drop=True)

rows = []

for number, pair in enumerate(
    near.itertuples(index=False),
    start=1
):

    a = lookup.loc[str(pair.id_a)]
    b = lookup.loc[str(pair.id_b)]

    rows.append({
        "pair_number": number,
        "pair_key": pair.pair_key,
        "similarity": float(pair.similarity),

        "id_a": str(pair.id_a),
        "case_a": str(a["case"]),
        "label_a": str(a["sentiment_label"]),
        "label_notes_a": str(a["label_notes"]),
        "text_original_a": str(a["text_original"]),
        "text_semantic_a": str(
            a["text_semantic_verified"]
        ),
        "text_linguistic_a": str(
            a["text_linguistic"]
        ),
        "final_text_hash_a": str(
            a["final_text_hash"]
        ),

        "id_b": str(pair.id_b),
        "case_b": str(b["case"]),
        "label_b": str(b["sentiment_label"]),
        "label_notes_b": str(b["label_notes"]),
        "text_original_b": str(b["text_original"]),
        "text_semantic_b": str(
            b["text_semantic_verified"]
        ),
        "text_linguistic_b": str(
            b["text_linguistic"]
        ),
        "final_text_hash_b": str(
            b["final_text_hash"]
        ),

        "label_disagreement": (
            "YES"
            if pair.label_a != pair.label_b
            else "NO"
        ),

        # Editable manual-review fields:
        "review_decision": "",
        "label_audit_status": "",
        "review_notes": "",
        "reviewer": "",
        "reviewed_at": ""
    })

review = pd.DataFrame(rows)

# The snapshot contains only immutable source fields.
editable_fields = [
    "review_decision",
    "label_audit_status",
    "review_notes",
    "reviewer",
    "reviewed_at"
]

snapshot = review.drop(
    columns=editable_fields
).copy()

assert len(review) == 12
assert review["pair_key"].is_unique

assert int(
    review["label_disagreement"].eq("YES").sum()
) == 3

print("REVIEW DATA PREPARED: PASS")

# ============================================================
# G. GUIDELINES
# ============================================================

guidelines = pd.DataFrame([
    {
        "Field": "OBJECTIVE",
        "Instruction": (
            "Review 12 near-duplicate candidate pairs "
            "without deleting tweets or changing labels."
        )
    },
    {
        "Field": "SAME_CONTENT",
        "Instruction": (
            "Both tweets convey essentially the same "
            "content or are close variants of the same "
            "news/post. Group them for leakage prevention."
        )
    },
    {
        "Field": "DISTINCT_CONTENT",
        "Instruction": (
            "Texts look similar, but their meaningful "
            "content/context differs. Explain the difference."
        )
    },
    {
        "Field": "UNRESOLVED",
        "Instruction": (
            "Evidence is insufficient. Requires another "
            "review before split."
        )
    },
    {
        "Field": "LABEL_CONSISTENT",
        "Instruction": (
            "Both labels are justified by the codebook. "
            "This can include legitimately different "
            "labels for distinct contexts."
        )
    },
    {
        "Field": "LABEL_REVIEW",
        "Instruction": (
            "At least one label potentially conflicts "
            "with codebook v1.1. Requires separate "
            "manual label adjudication."
        )
    },
    {
        "Field": "label_audit_status",
        "Instruction": (
            "Choose LABEL_CONSISTENT or LABEL_REVIEW. "
            "Do not edit label_a or label_b."
        )
    },
    {
        "Field": "review_notes",
        "Instruction": (
            "Explain the relation between texts and "
            "the author's stance toward national disaster "
            "status. Do not infer from keywords only."
        )
    },
    {
        "Field": "reviewer",
        "Instruction": "Name or reviewer code."
    },
    {
        "Field": "reviewed_at",
        "Instruction": (
            "Actual review date in YYYY-MM-DD format."
        )
    },
    {
        "Field": "IMMUTABLE",
        "Instruction": (
            "All columns except the final five manual "
            "review fields must remain unchanged."
        )
    },
    {
        "Field": "IMPORTANT",
        "Instruction": (
            "SAME_CONTENT does not authorize automatic "
            "relabeling. Keep label correction separate "
            "from leakage grouping."
        )
    }
])

# ============================================================
# H. PREPARE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Recheck to avoid overwriting prior reviews.
for path in [
    REVIEW_PATH,
    SNAPSHOT_PATH,
    MANIFEST_PATH
]:
    if path.exists():
        raise FileExistsError(
            f"Output exists: {path}"
        )

# ============================================================
# I. WRITE SNAPSHOT
# ============================================================

snapshot.to_csv(
    SNAPSHOT_PATH,
    index=False,
    encoding="utf-8-sig"
)

snapshot_hash = sha256_file(SNAPSHOT_PATH)

# ============================================================
# J. CREATE EXCEL WORKBOOK
# ============================================================

with pd.ExcelWriter(
    REVIEW_PATH,
    engine="openpyxl"
) as writer:

    guidelines.to_excel(
        writer,
        sheet_name="GUIDELINES",
        index=False
    )

    review.to_excel(
        writer,
        sheet_name="REVIEW_12",
        index=False
    )

    ws = writer.sheets["REVIEW_12"]
    guide_ws = writer.sheets["GUIDELINES"]

    ws.freeze_panes = "D2"
    ws.auto_filter.ref = ws.dimensions

    # Header formatting
    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    yellow_fill = PatternFill(
        fill_type="solid",
        fgColor="FFF2CC"
    )

    red_fill = PatternFill(
        fill_type="solid",
        fgColor="FCE4D6"
    )

    gray_fill = PatternFill(
        fill_type="solid",
        fgColor="E7E6E6"
    )

    for cell in ws[1]:
        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )
        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    ws.row_dimensions[1].height = 35

    for col_index, column in enumerate(
        review.columns,
        start=1
    ):

        column_letter = ws.cell(
            1, col_index
        ).column_letter

        if column in [
            "text_original_a",
            "text_original_b",
            "text_semantic_a",
            "text_semantic_b",
            "text_linguistic_a",
            "text_linguistic_b"
        ]:
            width = 65

        elif column in [
            "label_notes_a",
            "label_notes_b",
            "review_notes"
        ]:
            width = 55

        elif column in [
            "id_a",
            "id_b",
            "pair_key",
            "final_text_hash_a",
            "final_text_hash_b"
        ]:
            width = 35

        else:
            width = 24

        ws.column_dimensions[
            column_letter
        ].width = width

        for row_index in range(
            2,
            ws.max_row + 1
        ):
            cell = ws.cell(
                row_index,
                col_index
            )

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

            if column in [
                "id_a",
                "id_b",
                "pair_key"
            ]:
                cell.number_format = "@"

            if column in editable_fields:
                cell.fill = yellow_fill

            elif column not in [
                "pair_number",
                "similarity",
                "label_disagreement"
            ]:
                cell.fill = gray_fill

    # Highlight pairs with different labels.
    disagreement_col = (
        review.columns.get_loc(
            "label_disagreement"
        ) + 1
    )

    for row_index in range(
        2, ws.max_row + 1
    ):
        disagreement = ws.cell(
            row_index,
            disagreement_col
        )

        if disagreement.value == "YES":
            disagreement.fill = red_fill
            disagreement.font = Font(
                bold=True,
                color="9C0006"
            )

    # Manual decision dropdowns.
    decision_col = (
        review.columns.get_loc(
            "review_decision"
        ) + 1
    )

    label_audit_col = (
        review.columns.get_loc(
            "label_audit_status"
        ) + 1
    )

    decision_dv = DataValidation(
        type="list",
        formula1=(
            '"SAME_CONTENT,DISTINCT_CONTENT,UNRESOLVED"'
        ),
        allow_blank=False
    )

    label_dv = DataValidation(
        type="list",
        formula1=(
            '"LABEL_CONSISTENT,LABEL_REVIEW"'
        ),
        allow_blank=False
    )

    decision_dv.error = (
        "Choose a valid review decision."
    )
    decision_dv.errorTitle = "Invalid Decision"
    decision_dv.showErrorMessage = True

    label_dv.error = (
        "Choose a valid label audit status."
    )
    label_dv.errorTitle = "Invalid Label Audit"
    label_dv.showErrorMessage = True

    ws.add_data_validation(decision_dv)
    ws.add_data_validation(label_dv)

    decision_letter = ws.cell(
        1, decision_col
    ).column_letter

    label_letter = ws.cell(
        1, label_audit_col
    ).column_letter

    decision_dv.add(
        f"{decision_letter}2:"
        f"{decision_letter}{ws.max_row}"
    )

    label_dv.add(
        f"{label_letter}2:"
        f"{label_letter}{ws.max_row}"
    )

    # Guidelines formatting.
    guide_ws.freeze_panes = "A2"
    guide_ws.column_dimensions["A"].width = 28
    guide_ws.column_dimensions["B"].width = 105

    for cell in guide_ws[1]:
        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )
        cell.fill = header_fill

    for row in guide_ws.iter_rows(
        min_row=2
    ):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

# ============================================================
# K. MANIFEST & FINAL SAFETY CHECK
# ============================================================

source_hashes_after = {
    "modeling_dataset": sha256_file(SOURCE_DATA),
    "near_duplicate_source": sha256_file(NEAR_SOURCE),
    "preflight_report": sha256_file(PREFLIGHT_REPORT)
}

assert source_hashes_before == source_hashes_after, (
    "Source files changed during preparation."
)

manifest = {
    "stage": "04_Modeling",
    "part": "02 Near-Duplicate Review Preparation",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_commit": (
        "4bfe3d3780026a2be95938c40fe31193324fb7cc"
    ),
    "total_modeling_rows": 348,
    "review_pairs": int(len(review)),
    "different_label_pairs": 3,
    "near_duplicate_threshold": 0.80,
    "source_hashes": source_hashes_before,
    "snapshot_sha256": snapshot_hash,
    "review_workbook_sha256_at_creation": (
        sha256_file(REVIEW_PATH)
    ),
    "reviewed_pairs": 0,
    "labels_modified": False,
    "source_data_modified": False,
    "leakage_groups_created": False,
    "split_created": False,
    "training_started": False,
    "status": "AWAITING_MANUAL_NEAR_DUPLICATE_REVIEW"
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("MODELING 02 — FINAL REPORT")
print("=" * 70)

print("Total modeling tweets :", len(df))
print("Near-duplicate pairs  :", len(review))
print(
    "Different-label pairs :",
    int(
        review["label_disagreement"]
        .eq("YES").sum()
    )
)

print("\nReview workbook :", REVIEW_PATH)
print("Immutable snapshot:", SNAPSHOT_PATH)
print("Manifest         :", MANIFEST_PATH)

print("\nSOURCE INTEGRITY: PASS")
print("REVIEW WORKBOOK CREATED: PASS")
print("ORIGINAL LABELS: UNCHANGED")
print("LEAKAGE GROUPING: NOT STARTED")
print("TRAIN / TEST SPLIT: NOT STARTED")
print("TRAINING: HOLD")

print("\nMODELING 02 REVIEW PREPARATION: PASS")
print("=" * 70)


MODELING 02 — NEAR-DUPLICATE REVIEW PREPARATION
SOURCE INTEGRITY: PASS
PREFLIGHT MANIFEST: PASS
NO OUTPUT OVERWRITE: PASS
MODEL DATA: 348 ROWS — PASS
NEAR-DUPLICATE PAIRS: 12 — PASS
PAIR-TO-SOURCE VALIDATION: PASS
DIFFERENT LABEL PAIRS: 3
REVIEW DATA PREPARED: PASS

MODELING 02 — FINAL REPORT
Total modeling tweets : 348
Near-duplicate pairs  : 12
Different-label pairs : 3

Review workbook : C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\near_duplicate_review\near_duplicate_review_12.xlsx
Immutable snapshot: C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\near_duplicate_review\near_duplicate_snapshot.csv
Manifest         : C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\near_duplicate_review\near_duplicate_review_manifest.json

SOURCE INTEGRITY: PASS
REVIEW WORKBOOK CREATED: PASS
ORIGINAL LABELS: UNCHANGED
LEAKAGE GROUPING: NOT STARTED
TRAIN / TEST SPLIT: NOT STARTED
TRAINING: HOLD

MODELING 02 REVIEW PREPARATION: PASS


In [6]:
# MODELING 03 — NEAR-DUPLICATE REVIEW VALIDATION
# READ ONLY SOURCES | NO LABEL CHANGES | NO SPLIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

REVIEW_DIR = (
    ROOT / "04_Modeling" / "outputs"
    / "near_duplicate_review"
)

REVIEW_PATH = REVIEW_DIR / "near_duplicate_review_12.xlsx"
SNAPSHOT_PATH = REVIEW_DIR / "near_duplicate_snapshot.csv"
MANIFEST_PATH = REVIEW_DIR / "near_duplicate_review_manifest.json"

VALIDATION_PATH = REVIEW_DIR / "review_validation_report.json"
DECISIONS_PATH = REVIEW_DIR / "near_duplicate_review_decisions.csv"
LABEL_FLAGS_PATH = REVIEW_DIR / "label_review_flags.csv"

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def clean_col(series):
    return series.astype("string").fillna("").str.strip()

def compare_immutable(a, b, col):
    left = a.set_index("pair_key")[col]
    right = b.set_index("pair_key")[col]
    if col == "similarity":
        return (
            pd.to_numeric(left, errors="raise")
            .sort_index()
            .round(6)
            .equals(
                pd.to_numeric(right, errors="raise")
                .sort_index()
                .round(6)
            )
        )
    return clean_col(left).sort_index().equals(
        clean_col(right).sort_index()
    )

print("=" * 65)
print("MODELING 03 — NEAR-DUPLICATE REVIEW VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# A. CHECK INPUTS AND OUTPUT SAFETY
# ------------------------------------------------------------

for path in [REVIEW_PATH, SNAPSHOT_PATH, MANIFEST_PATH]:
    assert path.is_file(), f"Missing input: {path}"

for path in [
    VALIDATION_PATH,
    DECISIONS_PATH,
    LABEL_FLAGS_PATH
]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists, no overwrite allowed: {path}"
        )

manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

assert manifest["status"] == (
    "AWAITING_MANUAL_NEAR_DUPLICATE_REVIEW"
)

assert sha256_file(SNAPSHOT_PATH) == (
    manifest["snapshot_sha256"]
), "Snapshot changed. STOP."

input_hashes_before = {
    "review": sha256_file(REVIEW_PATH),
    "snapshot": sha256_file(SNAPSHOT_PATH),
    "manifest": sha256_file(MANIFEST_PATH)
}

# ------------------------------------------------------------
# B. LOAD REVIEW AND SNAPSHOT
# ------------------------------------------------------------

review = pd.read_excel(
    REVIEW_PATH,
    sheet_name="REVIEW_12",
    dtype={"id_a": "string", "id_b": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

snapshot = pd.read_csv(
    SNAPSHOT_PATH,
    dtype={"id_a": "string", "id_b": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

assert len(review) == len(snapshot) == 12

for frame in [review, snapshot]:
    for col in ["id_a", "id_b", "pair_key"]:
        frame[col] = clean_col(frame[col])

    assert frame["pair_key"].is_unique
    assert frame["id_a"].ne("").all()
    assert frame["id_b"].ne("").all()

assert set(review["pair_key"]) == set(snapshot["pair_key"])

print("REVIEW ROW COUNT: PASS")
print("PAIR ID INTEGRITY: PASS")

# ------------------------------------------------------------
# C. VERIFY IMMUTABLE FIELDS
# ------------------------------------------------------------

editable = {
    "review_decision",
    "label_audit_status",
    "review_notes",
    "reviewer",
    "reviewed_at"
}

immutable = [
    col for col in snapshot.columns
    if col != "pair_key"
]

assert set(immutable).issubset(review.columns)
assert editable.issubset(review.columns)

for col in immutable:
    assert compare_immutable(review, snapshot, col), (
        f"IMMUTABLE FIELD CHANGED: {col}"
    )

print("IMMUTABLE SOURCE FIELDS: PASS")

# ------------------------------------------------------------
# D. NORMALIZE AND VALIDATE DECISIONS
# ------------------------------------------------------------

for col in ["review_decision", "label_audit_status"]:
    review[col] = clean_col(review[col]).str.upper()

for col in ["review_notes", "reviewer"]:
    review[col] = clean_col(review[col])

VALID_DECISIONS = {
    "SAME_CONTENT",
    "DISTINCT_CONTENT",
    "UNRESOLVED"
}

VALID_LABEL_AUDIT = {
    "LABEL_CONSISTENT",
    "LABEL_REVIEW"
}

assert review["review_decision"].isin(
    VALID_DECISIONS
).all(), (
    "Invalid or missing review_decision. "
    "Check dropdown values in Excel."
)

assert review["label_audit_status"].isin(
    VALID_LABEL_AUDIT
).all(), (
    "Invalid or missing label_audit_status."
)

assert review["review_notes"].ne("").all(), (
    "Missing review_notes."
)

assert review["reviewer"].ne("").all(), (
    "Missing reviewer."
)

date_strings = clean_col(review["reviewed_at"])

review_dates = pd.to_datetime(
    date_strings,
    errors="coerce",
    format="mixed"
)

assert review_dates.notna().all(), (
    "Missing or invalid reviewed_at."
)

review["reviewed_at"] = (
    review_dates.dt.strftime("%Y-%m-%d")
)

print("ALL 12 REVIEW DECISIONS: VALID")
print("REVIEW METADATA: COMPLETE")

# ------------------------------------------------------------
# E. CLASSIFY NEXT ACTION
# ------------------------------------------------------------

review["label_disagreement"] = (
    clean_col(review["label_disagreement"]).str.upper()
)

different_labels = review[
    review["label_disagreement"] == "YES"
]

assert len(different_labels) == 3

unresolved = review[
    review["review_decision"] == "UNRESOLVED"
]

label_review = review[
    review["label_audit_status"] == "LABEL_REVIEW"
]

same_content_label_conflicts = review[
    (review["review_decision"] == "SAME_CONTENT")
    & (review["label_disagreement"] == "YES")
]

# A pair can require label review even when the original
# labels agree, if both original labels are suspected wrong.

counts = {
    "SAME_CONTENT": int(
        review["review_decision"].eq("SAME_CONTENT").sum()
    ),
    "DISTINCT_CONTENT": int(
        review["review_decision"].eq("DISTINCT_CONTENT").sum()
    ),
    "UNRESOLVED": len(unresolved),
    "LABEL_CONSISTENT": int(
        review["label_audit_status"].eq(
            "LABEL_CONSISTENT"
        ).sum()
    ),
    "LABEL_REVIEW": len(label_review)
}

# Prevent premature split when review is incomplete
# or when adjudication is still necessary.
needs_adjudication = (
    len(label_review) > 0
    or len(same_content_label_conflicts) > 0
)

if len(unresolved) > 0:
    status = "HOLD_UNRESOLVED_PAIRS"
elif needs_adjudication:
    status = "HOLD_LABEL_ADJUDICATION"
else:
    status = "READY_FOR_LEAKAGE_GROUPING"

print("\nREVIEW DECISION DISTRIBUTION:")
display(
    review["review_decision"].value_counts()
)

print("\nLABEL AUDIT DISTRIBUTION:")
display(
    review["label_audit_status"].value_counts()
)

print("\nTHREE ORIGINAL LABEL-DISAGREEMENT PAIRS:")
display(
    different_labels[
        [
            "pair_number", "id_a", "id_b",
            "label_a", "label_b",
            "review_decision", "label_audit_status",
            "review_notes"
        ]
    ]
)

if len(label_review):
    print("\nLABEL REVIEW FLAGS:")
    display(
        label_review[
            [
                "pair_number", "id_a", "id_b",
                "label_a", "label_b",
                "review_decision", "review_notes"
            ]
        ]
    )

# ------------------------------------------------------------
# F. WRITE AUDIT OUTPUTS
# ------------------------------------------------------------

report = {
    "stage": "04_Modeling",
    "part": "03 Near-Duplicate Review Validation",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_commit": (
        "aca08c7abb47643b6c5e35fa82f5d51bc1d8a545"
    ),
    "total_pairs": len(review),
    "original_label_disagreement_pairs": 3,
    "decision_counts": counts,
    "same_content_with_different_labels": int(
        len(same_content_label_conflicts)
    ),
    "unresolved_pairs": len(unresolved),
    "label_review_flags": len(label_review),
    "dataset_labels_modified": False,
    "leakage_groups_created": False,
    "split_created": False,
    "training_started": False,
    "status": status,
    "source_hashes": input_hashes_before
}

# Verify inputs stayed unchanged.
assert input_hashes_before == {
    "review": sha256_file(REVIEW_PATH),
    "snapshot": sha256_file(SNAPSHOT_PATH),
    "manifest": sha256_file(MANIFEST_PATH)
}

review.to_csv(
    DECISIONS_PATH,
    index=False,
    encoding="utf-8-sig"
)

label_review.to_csv(
    LABEL_FLAGS_PATH,
    index=False,
    encoding="utf-8-sig"
)

VALIDATION_PATH.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("MODELING 03 — VALIDATION SUMMARY")
print("=" * 65)

print("Reviewed pairs             :", len(review))
print("SAME_CONTENT               :", counts["SAME_CONTENT"])
print("DISTINCT_CONTENT           :", counts["DISTINCT_CONTENT"])
print("UNRESOLVED                 :", counts["UNRESOLVED"])
print("LABEL_REVIEW               :", counts["LABEL_REVIEW"])
print(
    "Same-content label conflicts:",
    len(same_content_label_conflicts)
)

print("\nStatus:", status)
print("Report:", VALIDATION_PATH)
print("Decisions:", DECISIONS_PATH)
print("Label flags:", LABEL_FLAGS_PATH)

print("\nMODELING 03 REVIEW VALIDATION: PASS")
print("DATASET LABELS: UNCHANGED")
print("TRAINING: HOLD")


MODELING 03 — NEAR-DUPLICATE REVIEW VALIDATION
REVIEW ROW COUNT: PASS
PAIR ID INTEGRITY: PASS
IMMUTABLE SOURCE FIELDS: PASS
ALL 12 REVIEW DECISIONS: VALID
REVIEW METADATA: COMPLETE

REVIEW DECISION DISTRIBUTION:


review_decision
SAME_CONTENT        11
DISTINCT_CONTENT     1
Name: count, dtype: Int64


LABEL AUDIT DISTRIBUTION:


label_audit_status
LABEL_CONSISTENT    10
LABEL_REVIEW         2
Name: count, dtype: Int64


THREE ORIGINAL LABEL-DISAGREEMENT PAIRS:


,pair_number,id_a,id_b,label_a,label_b,review_decision,label_audit_status,review_notes
4,5,1027727426883805184,1027914286184550402,NETRAL,POSITIF,DISTINCT_CONTENT,LABEL_REVIEW,hanya menyampaikan informasi
8,9,1046691986596495360,1046787140594618373,NETRAL,POSITIF,SAME_CONTENT,LABEL_REVIEW,hanya menyampaika pendapat dari tokoh lain
9,10,1046064035505078274,1046217614148624387,POSITIF,NETRAL,SAME_CONTENT,LABEL_CONSISTENT,hanya menyampaikan informasi



LABEL REVIEW FLAGS:


,pair_number,id_a,id_b,label_a,label_b,review_decision,review_notes
4,5,1027727426883805184,1027914286184550402,NETRAL,POSITIF,DISTINCT_CONTENT,hanya menyampaikan informasi
8,9,1046691986596495360,1046787140594618373,NETRAL,POSITIF,SAME_CONTENT,hanya menyampaika pendapat dari tokoh lain



MODELING 03 — VALIDATION SUMMARY
Reviewed pairs             : 12
SAME_CONTENT               : 11
DISTINCT_CONTENT           : 1
UNRESOLVED                 : 0
LABEL_REVIEW               : 2
Same-content label conflicts: 2

Status: HOLD_LABEL_ADJUDICATION
Report: C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\near_duplicate_review\review_validation_report.json
Decisions: C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\near_duplicate_review\near_duplicate_review_decisions.csv
Label flags: C:\Penelitian Sentimen Bencana Nasional\04_Modeling\outputs\near_duplicate_review\label_review_flags.csv

MODELING 03 REVIEW VALIDATION: PASS
DATASET LABELS: UNCHANGED
TRAINING: HOLD
